# Classificação em Tempo Real por Repetição

Este notebook faz inferência em tempo real usando os checkpoints `model_fold*.pth` treinados no `05-lstm.ipynb`.

Fluxo:
1. Lê frames da webcam ou vídeo.
2. Extrai landmarks com MediaPipe Pose.
3. Segmenta cada repetição por máquina de estados do ângulo do cotovelo.
4. Monta a sequência da repetição com o mesmo pré-processamento do treino.
5. Faz predição com ensemble dos folds e classifica `fatigado`/`não fatigado` a cada repetição.

> Sim: é possível (e recomendado) usar os `model_fold*.pth` para ensemble em inferência.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mediapipe as mp


# ----------------------------
# Configuração
# ----------------------------
RESULTS_DIR = Path("saida") / "resultados"
MODEL_PATTERN = "model_fold*.pth"
VIDEO_SOURCE = Path(r"entrada") / "my_record_01.mp4"  # 0 = webcam | ou caminho de vídeo, ex: "entrada/video.mp4"
ACTIVE_SIDE = "RIGHT"  # "LEFT" ou "RIGHT"
EXPECTED_TOTAL_REPS = 12  # usado para feature rep_progress

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REP_META_COLS = ["rep_duration", "rom_deg", "elbow_start_deg", "elbow_peak_deg", "mean_vis_arm"]


# ----------------------------
# Modelo (mesmo do treino)
# ----------------------------
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1, bias=False),
        )

    def forward(self, lstm_out: torch.Tensor):
        scores = self.attn(lstm_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights


class FatigueNet(nn.Module):
    def __init__(
        self,
        seq_input_dim: int,
        ctx_input_dim: int,
        hidden_dim: int = 128,
        num_layers: int = 2,
        dropout: float = 0.4,
        ctx_hidden: int = 64,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=seq_input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.attention = TemporalAttention(hidden_dim * 2)
        self.seq_norm = nn.LayerNorm(hidden_dim * 2)

        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_input_dim, ctx_hidden),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(ctx_hidden, ctx_hidden // 2),
            nn.ReLU(),
        )

        fusion_dim = hidden_dim * 2 + ctx_hidden // 2
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq: torch.Tensor, x_ctx: torch.Tensor):
        lstm_out, _ = self.lstm(x_seq)
        attn_out, _ = self.attention(lstm_out)
        attn_out = self.seq_norm(attn_out)
        ctx_out = self.ctx_mlp(x_ctx)
        fused = torch.cat([attn_out, ctx_out], dim=1)
        logits = self.classifier(fused).squeeze(1)
        return logits


def infer_arch_from_state_dict(state_dict: Dict[str, torch.Tensor]) -> Dict[str, int]:
    hidden_dim = state_dict["lstm.weight_ih_l0"].shape[0] // 4
    seq_input_dim = state_dict["lstm.weight_ih_l0"].shape[1]

    num_layers = 0
    while f"lstm.weight_ih_l{num_layers}" in state_dict:
        num_layers += 1

    ctx_hidden = state_dict["ctx_mlp.0.weight"].shape[0]
    ctx_input_dim = state_dict["ctx_mlp.0.weight"].shape[1]

    return {
        "hidden_dim": hidden_dim,
        "seq_input_dim": seq_input_dim,
        "num_layers": num_layers,
        "ctx_hidden": ctx_hidden,
        "ctx_input_dim": ctx_input_dim,
    }


# ----------------------------
# Pré-processamento (igual ao treino)
# ----------------------------
def resample_to_T(X: np.ndarray, T: int) -> np.ndarray:
    n, f = X.shape
    if n == T:
        return X.astype(np.float32)
    if n < 2:
        return np.repeat(X, T, axis=0).astype(np.float32)

    old = np.linspace(0, 1, n)
    new = np.linspace(0, 1, T)
    Xr = np.zeros((T, f), dtype=np.float32)
    for j in range(f):
        Xr[:, j] = np.interp(new, old, X[:, j])
    return Xr


def swap_lr_landmarks(X: np.ndarray, col_index: Dict[str, int], lmA: int, lmB: int):
    for suffix in ["x", "y", "z", "vis"]:
        a = f"lm{lmA}_{suffix}"
        b = f"lm{lmB}_{suffix}"
        if a in col_index and b in col_index:
            ia, ib = col_index[a], col_index[b]
            X[:, [ia, ib]] = X[:, [ib, ia]]


def canonicalize_side(X: np.ndarray, col_index: Dict[str, int], side: Optional[str]):
    if side is None:
        return X
    side = str(side).upper()
    if side != "RIGHT":
        return X

    for lm in [11, 12, 13, 14, 15, 16, 23, 24]:
        k = f"lm{lm}_x"
        if k in col_index:
            X[:, col_index[k]] = 1.0 - X[:, col_index[k]]

    swap_lr_landmarks(X, col_index, 11, 12)
    swap_lr_landmarks(X, col_index, 13, 14)
    swap_lr_landmarks(X, col_index, 15, 16)
    swap_lr_landmarks(X, col_index, 23, 24)
    return X


def normalize_body_framewise_xyz(X: np.ndarray, col_index: Dict[str, int]):
    required = ["lm23_x", "lm24_x", "lm23_y", "lm24_y", "lm23_z", "lm24_z", "lm11_x", "lm12_x", "lm11_y", "lm12_y", "lm11_z", "lm12_z"]
    if any(k not in col_index for k in required):
        return X

    hx = (X[:, col_index["lm23_x"]] + X[:, col_index["lm24_x"]]) / 2
    hy = (X[:, col_index["lm23_y"]] + X[:, col_index["lm24_y"]]) / 2
    hz = (X[:, col_index["lm23_z"]] + X[:, col_index["lm24_z"]]) / 2

    dx = X[:, col_index["lm11_x"]] - X[:, col_index["lm12_x"]]
    dy = X[:, col_index["lm11_y"]] - X[:, col_index["lm12_y"]]
    dz = X[:, col_index["lm11_z"]] - X[:, col_index["lm12_z"]]
    scale = np.sqrt(dx * dx + dy * dy + dz * dz)
    scale = np.clip(scale, 1e-3, None)

    for lm in [11, 12, 13, 14, 15, 16, 23, 24]:
        for ax, h in [("x", hx), ("y", hy), ("z", hz)]:
            k = f"lm{lm}_{ax}"
            if k in col_index:
                X[:, col_index[k]] = (X[:, col_index[k]] - h) / scale
    return X


def add_velocity_acceleration(X: np.ndarray) -> np.ndarray:
    vel = np.zeros_like(X)
    acc = np.zeros_like(X)
    vel[1:] = X[1:] - X[:-1]
    acc[2:] = vel[2:] - vel[1:-1]
    return np.concatenate([X, vel, acc], axis=1)


def build_rep_sequence_from_rows(rep_rows: List[Dict[str, float]], feature_cols: List[str], T: int, col_index: Dict[str, int], side: str):
    rep_df = pd.DataFrame(rep_rows)
    for c in feature_cols:
        if c not in rep_df.columns:
            rep_df[c] = np.nan

    feats = rep_df[feature_cols].apply(pd.to_numeric, errors="coerce")
    feats = feats.interpolate(limit_direction="both").fillna(0.0)

    X = feats.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    X = resample_to_T(X, T)
    X = canonicalize_side(X, col_index, side)
    X = normalize_body_framewise_xyz(X, col_index)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    X = add_velocity_acceleration(X)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X


print(f"Device: {DEVICE}")

Device: cuda


In [2]:
# ----------------------------
# Extração de landmarks + segmentação de repetição
# ----------------------------
def point_from_lm(lms, idx: int):
    p = lms[idx]
    return np.array([p.x, p.y, p.z], dtype=np.float32), float(p.visibility)


def elbow_angle_deg(shoulder: np.ndarray, elbow: np.ndarray, wrist: np.ndarray) -> float:
    v1 = shoulder - elbow
    v2 = wrist - elbow
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-8 or n2 < 1e-8:
        return np.nan
    cosang = np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))


@dataclass
class RepSegmenter:
    down_angle: float = 150.0
    up_angle: float = 70.0
    hysteresis: float = 6.0
    min_frames: int = 8
    max_frames: int = 240
    min_rom_for_valid_rep: float = 10.0

    state: str = "ready"
    rep_start_idx: Optional[int] = None
    start_angle: float = np.nan
    peak_angle: float = np.nan
    calibrated: bool = False

    def calibrate_from_angles(self, angles: List[float]):
        valid = np.array([a for a in angles if not np.isnan(a)], dtype=np.float32)
        if len(valid) < 12:
            return

        p10 = float(np.percentile(valid, 10))
        p90 = float(np.percentile(valid, 90))
        span = p90 - p10

        if span >= 20.0:
            self.up_angle = p10 + 0.25 * span
            self.down_angle = p10 + 0.80 * span

            self.up_angle = float(np.clip(self.up_angle, 30.0, 130.0))
            self.down_angle = float(np.clip(self.down_angle, 80.0, 175.0))

            if self.down_angle - self.up_angle < 15.0:
                self.down_angle = min(175.0, self.up_angle + 15.0)

        self.calibrated = True

    def update(self, angle: float, frame_idx: int) -> Optional[Dict[str, float]]:
        if np.isnan(angle):
            return None

        if self.state == "ready":
            if angle < (self.down_angle - self.hysteresis):
                self.state = "going_up"
                self.rep_start_idx = frame_idx
                self.start_angle = angle
                self.peak_angle = angle
            return None

        if self.state == "going_up":
            self.peak_angle = min(self.peak_angle, angle)
            if angle <= self.up_angle:
                self.state = "going_down"
            return None

        if self.state == "going_down":
            self.peak_angle = min(self.peak_angle, angle)
            if angle >= self.down_angle:
                end_idx = frame_idx
                start_idx = self.rep_start_idx if self.rep_start_idx is not None else frame_idx
                n_frames = end_idx - start_idx + 1
                self.state = "ready"

                if self.min_frames <= n_frames <= self.max_frames:
                    elbow_start = float(self.start_angle)
                    elbow_peak = float(self.peak_angle)
                    rom_deg = float(max(0.0, elbow_start - elbow_peak))

                    if rom_deg >= self.min_rom_for_valid_rep:
                        return {
                            "start_idx": int(start_idx),
                            "end_idx": int(end_idx),
                            "elbow_start_deg": elbow_start,
                            "elbow_peak_deg": elbow_peak,
                            "rom_deg": rom_deg,
                        }
            return None

        return None


def rep_context_vector(
    context_features: List[str],
    rep_history: List[Dict[str, float]],
    expected_total_reps: int,
) -> np.ndarray:
    cur = rep_history[-1]
    prev = rep_history[-2] if len(rep_history) >= 2 else None

    vals: Dict[str, float] = {}
    vals["rep_duration"] = float(cur.get("rep_duration", 0.0))
    vals["rom_deg"] = float(cur.get("rom_deg", 0.0))
    vals["elbow_start_deg"] = float(cur.get("elbow_start_deg", 0.0))
    vals["elbow_peak_deg"] = float(cur.get("elbow_peak_deg", 0.0))
    vals["mean_vis_arm"] = float(cur.get("mean_vis_arm", 0.0))

    total_reps = len(rep_history)
    vals["total_reps"] = float(total_reps)
    if expected_total_reps > 0:
        vals["rep_progress"] = float(np.clip(total_reps / expected_total_reps, 0.0, 1.0))
    else:
        vals["rep_progress"] = 0.0

    for c in REP_META_COLS:
        cur_v = float(vals.get(c, 0.0))
        prev_v = float(prev[c]) if prev is not None and c in prev else cur_v
        vals[f"delta_{c}"] = cur_v - prev_v

        denom = prev_v if abs(prev_v) > 1e-6 else 1e-6
        vals[f"rel_{c}"] = (cur_v - prev_v) / denom

        last3 = [float(x.get(c, 0.0)) for x in rep_history[-3:]]
        vals[f"rolling3_{c}"] = float(np.mean(last3)) if len(last3) else 0.0

    return np.array([float(vals.get(c, 0.0)) for c in context_features], dtype=np.float32)


def load_ensemble_models(results_dir: Path, model_pattern: str = "model_fold*.pth"):
    ckpt_paths = sorted(results_dir.glob(model_pattern))
    if not ckpt_paths:
        raise FileNotFoundError(f"Nenhum checkpoint encontrado em {results_dir} com padrão {model_pattern}")

    folds = []
    for p in ckpt_paths:
        ckpt = torch.load(p, map_location=DEVICE)
        arch = infer_arch_from_state_dict(ckpt["state_dict"])

        model = FatigueNet(
            seq_input_dim=arch["seq_input_dim"],
            ctx_input_dim=arch["ctx_input_dim"],
            hidden_dim=arch["hidden_dim"],
            num_layers=arch["num_layers"],
            ctx_hidden=arch["ctx_hidden"],
            dropout=0.0,
        ).to(DEVICE)
        model.load_state_dict(ckpt["state_dict"])
        model.eval()

        folds.append(
            {
                "path": p,
                "model": model,
                "threshold": float(ckpt.get("threshold", 0.5)),
                "seq_mean": np.array(ckpt["seq_mean"], dtype=np.float32),
                "seq_std": np.array(ckpt["seq_std"], dtype=np.float32),
                "ctx_mean": np.array(ckpt["ctx_mean"], dtype=np.float32),
                "ctx_std": np.array(ckpt["ctx_std"], dtype=np.float32),
                "T": int(ckpt["T"]),
                "feature_cols": list(ckpt["feature_cols"]),
                "context_features": list(ckpt["context_features"]),
            }
        )

    base = folds[0]
    for f in folds[1:]:
        assert f["feature_cols"] == base["feature_cols"], "feature_cols inconsistentes entre folds"
        assert f["context_features"] == base["context_features"], "context_features inconsistentes entre folds"
        assert f["T"] == base["T"], "T inconsistente entre folds"

    print(f"Ensemble carregado: {len(folds)} modelos")
    print(f"T={base['T']} | n_features={len(base['feature_cols'])} | n_context={len(base['context_features'])}")
    return folds


def predict_rep_ensemble(
    folds,
    seq_raw: np.ndarray,
    ctx_raw: np.ndarray,
) -> Dict[str, float]:
    probs = []
    fold_preds = []

    for f in folds:
        X = (seq_raw - f["seq_mean"]) / (f["seq_std"] + 1e-8)
        C = (ctx_raw - f["ctx_mean"]) / (f["ctx_std"] + 1e-8)

        x_t = torch.from_numpy(X).unsqueeze(0).to(DEVICE)
        c_t = torch.from_numpy(C).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logit = f["model"](x_t, c_t)
            prob = torch.sigmoid(logit).item()

        probs.append(prob)
        fold_preds.append(int(prob >= f["threshold"]))

    prob_mean = float(np.mean(probs))
    thr_mean = float(np.mean([f["threshold"] for f in folds]))
    label_by_prob = int(prob_mean >= thr_mean)
    label_by_vote = int(np.mean(fold_preds) >= 0.5)

    return {
        "prob_mean": prob_mean,
        "thr_mean": thr_mean,
        "label_prob": label_by_prob,
        "label_vote": label_by_vote,
    }

In [3]:
# ----------------------------
# Loop em tempo real
# ----------------------------
def _fit_frame_for_screen(frame: np.ndarray, max_w: int = 1280, max_h: int = 720) -> np.ndarray:
    h, w = frame.shape[:2]
    scale = min(max_w / max(w, 1), max_h / max(h, 1), 1.0)
    if scale >= 1.0:
        return frame
    out_w = max(1, int(w * scale))
    out_h = max(1, int(h * scale))
    return cv2.resize(frame, (out_w, out_h), interpolation=cv2.INTER_AREA)


def run_realtime_classifier(
    video_source=VIDEO_SOURCE,
    active_side: str = ACTIVE_SIDE,
    expected_total_reps: int = EXPECTED_TOTAL_REPS,
    calibration_seconds: float = 2.0,
    smoothing_alpha: float = 0.30,
    display_max_w: int = 1280,
    display_max_h: int = 720,
    export_csv: bool = True,
    csv_tag: str = "realtime",
):
    folds = load_ensemble_models(RESULTS_DIR, MODEL_PATTERN)
    base = folds[0]

    feature_cols = base["feature_cols"]
    context_features = base["context_features"]
    T = base["T"]
    col_index = {c: i for i, c in enumerate(feature_cols)}

    cap = cv2.VideoCapture(str(video_source))
    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir fonte de vídeo: {video_source}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-6:
        fps = 30.0

    calib_frames = max(1, int(calibration_seconds * fps))

    mp_pose = mp.solutions.pose
    mp_draw = mp.solutions.drawing_utils

    segmenter = RepSegmenter()
    rep_history: List[Dict[str, float]] = []

    frame_rows: List[Dict[str, float]] = []
    calib_angles: List[float] = []
    rep_metrics_rows: List[Dict[str, float]] = []

    angle_smooth = np.nan
    last_prob = None
    last_label = None
    rep_counter = 0

    window_name = "Real-time Fatigue Classifier"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(frame_rgb)

            row: Dict[str, float] = {"frame": frame_idx}
            angle_raw = np.nan

            if res.pose_landmarks:
                lms = res.pose_landmarks.landmark

                for i in range(33):
                    row[f"lm{i}_x"] = float(lms[i].x)
                    row[f"lm{i}_y"] = float(lms[i].y)
                    row[f"lm{i}_z"] = float(lms[i].z)
                    row[f"lm{i}_vis"] = float(lms[i].visibility)

                if str(active_side).upper() == "RIGHT":
                    shoulder, _ = point_from_lm(lms, 12)
                    elbow, _ = point_from_lm(lms, 14)
                    wrist, _ = point_from_lm(lms, 16)
                    vis_arm = np.mean([lms[12].visibility, lms[14].visibility, lms[16].visibility])
                else:
                    shoulder, _ = point_from_lm(lms, 11)
                    elbow, _ = point_from_lm(lms, 13)
                    wrist, _ = point_from_lm(lms, 15)
                    vis_arm = np.mean([lms[11].visibility, lms[13].visibility, lms[15].visibility])

                angle_raw = elbow_angle_deg(shoulder, elbow, wrist)
                row["mean_vis_arm"] = float(vis_arm)

                mp_draw.draw_landmarks(frame, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            else:
                row["mean_vis_arm"] = 0.0

            if np.isnan(angle_raw):
                angle = np.nan
            elif np.isnan(angle_smooth):
                angle_smooth = angle_raw
                angle = angle_smooth
            else:
                angle_smooth = (1.0 - smoothing_alpha) * angle_smooth + smoothing_alpha * angle_raw
                angle = angle_smooth

            frame_rows.append(row)

            is_calibrating = (not segmenter.calibrated) and (frame_idx < calib_frames)
            if is_calibrating:
                if not np.isnan(angle):
                    calib_angles.append(float(angle))
                if frame_idx == calib_frames - 1:
                    segmenter.calibrate_from_angles(calib_angles)
                    print(
                        f"Calibração: up={segmenter.up_angle:.1f} | down={segmenter.down_angle:.1f} | "
                        f"amostras={len(calib_angles)}"
                    )
            else:
                if not segmenter.calibrated:
                    segmenter.calibrate_from_angles(calib_angles)
                event = segmenter.update(angle, frame_idx)
                if event is not None:
                    s = int(event["start_idx"])
                    e = int(event["end_idx"])
                    rep_rows = frame_rows[s : e + 1]

                    rep_duration = (e - s + 1) / float(fps)
                    mean_vis = float(np.mean([r.get("mean_vis_arm", 0.0) for r in rep_rows]))

                    rep_meta = {
                        "rep_duration": float(rep_duration),
                        "rom_deg": float(event["rom_deg"]),
                        "elbow_start_deg": float(event["elbow_start_deg"]),
                        "elbow_peak_deg": float(event["elbow_peak_deg"]),
                        "mean_vis_arm": mean_vis,
                    }

                    rep_history.append(rep_meta)
                    rep_counter += 1

                    seq_raw = build_rep_sequence_from_rows(rep_rows, feature_cols, T, col_index, active_side)
                    ctx_raw = rep_context_vector(context_features, rep_history, expected_total_reps)

                    pred = predict_rep_ensemble(folds, seq_raw, ctx_raw)
                    last_prob = pred["prob_mean"]
                    last_label = pred["label_prob"]

                    txt = "FADIGADO" if last_label == 1 else "OK"
                    print(
                        f"Rep {rep_counter:02d} | prob={last_prob:.3f} | thr={pred['thr_mean']:.3f} | "
                        f"label={txt} | rom={rep_meta['rom_deg']:.1f} | dur={rep_meta['rep_duration']:.2f}s"
                    )

                    rep_metrics_rows.append(
                        {
                            "rep_id": rep_counter,
                            "start_frame": s,
                            "end_frame": e,
                            "rep_duration": rep_meta["rep_duration"],
                            "rom_deg": rep_meta["rom_deg"],
                            "elbow_start_deg": rep_meta["elbow_start_deg"],
                            "elbow_peak_deg": rep_meta["elbow_peak_deg"],
                            "mean_vis_arm": rep_meta["mean_vis_arm"],
                            "prob_mean": pred["prob_mean"],
                            "thr_mean": pred["thr_mean"],
                            "label_prob": pred["label_prob"],
                            "label_vote": pred["label_vote"],
                            "label_text": txt,
                        }
                    )

            hud1 = f"Reps: {rep_counter}"
            hud2 = f"Elbow angle: {angle:.1f}" if not np.isnan(angle) else "Elbow angle: --"

            if not segmenter.calibrated:
                hud3 = f"Calibrando... ({min(frame_idx + 1, calib_frames)}/{calib_frames})"
            elif last_prob is None:
                hud3 = "Ultima rep: --"
            else:
                label_txt = "FADIGADO" if last_label == 1 else "OK"
                hud3 = f"Ultima rep: {label_txt} ({last_prob:.2f})"

            hud4 = f"thr_up={segmenter.up_angle:.1f} thr_down={segmenter.down_angle:.1f}"

            cv2.putText(frame, hud1, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(frame, hud2, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(frame, hud3, (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(frame, hud4, (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 2)

            show = _fit_frame_for_screen(frame, max_w=display_max_w, max_h=display_max_h)
            cv2.imshow(window_name, show)
            key = cv2.waitKey(1) & 0xFF
            if key in (27, ord("q")):
                break

            frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    csv_paths = {}
    if export_csv:
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

        details_path = RESULTS_DIR / f"{csv_tag}_metrics_{ts}.csv"
        details_df = pd.DataFrame(rep_metrics_rows)
        details_df.to_csv(details_path, index=False)
        csv_paths["details"] = str(details_path)

        if len(details_df) > 0:
            summary_df = pd.DataFrame(
                [
                    {
                        "n_reps": int(len(details_df)),
                        "n_fatigado": int((details_df["label_prob"] == 1).sum()),
                        "n_ok": int((details_df["label_prob"] == 0).sum()),
                        "fatigue_rate": float((details_df["label_prob"] == 1).mean()),
                        "prob_mean_global": float(details_df["prob_mean"].mean()),
                        "prob_std_global": float(details_df["prob_mean"].std(ddof=0)),
                        "rom_mean_deg": float(details_df["rom_deg"].mean()),
                        "rom_std_deg": float(details_df["rom_deg"].std(ddof=0)),
                        "duration_mean_s": float(details_df["rep_duration"].mean()),
                        "duration_std_s": float(details_df["rep_duration"].std(ddof=0)),
                    }
                ]
            )
        else:
            summary_df = pd.DataFrame(
                [
                    {
                        "n_reps": 0,
                        "n_fatigado": 0,
                        "n_ok": 0,
                        "fatigue_rate": np.nan,
                        "prob_mean_global": np.nan,
                        "prob_std_global": np.nan,
                        "rom_mean_deg": np.nan,
                        "rom_std_deg": np.nan,
                        "duration_mean_s": np.nan,
                        "duration_std_s": np.nan,
                    }
                ]
            )

        summary_path = RESULTS_DIR / f"{csv_tag}_summary_{ts}.csv"
        summary_df.to_csv(summary_path, index=False)
        csv_paths["summary"] = str(summary_path)

        print(f"CSV detalhado salvo em: {details_path}")
        print(f"CSV resumo salvo em: {summary_path}")

    return {
        "n_reps": rep_counter,
        "csv_paths": csv_paths,
    }


# Exemplo 1: webcam
# run_realtime_classifier(video_source=0, active_side="RIGHT", expected_total_reps=12)

# Exemplo 2: vídeo de arquivo
# run_realtime_classifier(video_source="entrada/meu_video.mp4", active_side="RIGHT", expected_total_reps=12)

In [4]:
run_realtime_classifier(video_source="entrada/my_record_05.mp4", active_side="RIGHT", expected_total_reps=12)

C:\Users\Gush\AppData\Local\Temp\ipykernel_5956\1648703810.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(p, map_location=DEVICE)


Ensemble carregado: 5 modelos
T=70 | n_features=32 | n_context=22


c:\Users\Gush\Documents\USP\TESE\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Calibração: up=74.4 | down=119.6 | amostras=60
Rep 01 | prob=0.195 | thr=0.590 | label=OK | rom=52.2 | dur=2.07s
Rep 02 | prob=0.231 | thr=0.590 | label=OK | rom=57.4 | dur=2.40s
Rep 03 | prob=0.277 | thr=0.590 | label=OK | rom=58.5 | dur=2.00s
Rep 04 | prob=0.323 | thr=0.590 | label=OK | rom=53.1 | dur=4.43s
Rep 05 | prob=0.344 | thr=0.590 | label=OK | rom=63.5 | dur=2.03s
Rep 06 | prob=0.458 | thr=0.590 | label=OK | rom=58.3 | dur=6.03s
Rep 07 | prob=0.426 | thr=0.590 | label=OK | rom=64.6 | dur=1.67s
Rep 08 | prob=0.499 | thr=0.590 | label=OK | rom=58.4 | dur=1.93s
Rep 09 | prob=0.488 | thr=0.590 | label=OK | rom=60.5 | dur=1.77s
Rep 10 | prob=0.588 | thr=0.590 | label=OK | rom=60.5 | dur=1.93s
Rep 11 | prob=0.689 | thr=0.590 | label=FADIGADO | rom=61.2 | dur=2.17s
Rep 12 | prob=0.733 | thr=0.590 | label=FADIGADO | rom=60.0 | dur=2.13s
CSV detalhado salvo em: saida\resultados\realtime_metrics_20260419_181908.csv
CSV resumo salvo em: saida\resultados\realtime_summary_20260419_181908.

{'n_reps': 12,
 'csv_paths': {'details': 'saida\\resultados\\realtime_metrics_20260419_181908.csv',
  'summary': 'saida\\resultados\\realtime_summary_20260419_181908.csv'}}